# Practice 45 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — diamonds

Load `sns.load_dataset('diamonds')`.

The `clarity` column has values like `'SI2'`, `'VVS1'`, `'IF'` — a letter grade followed by an optional number.

- Extract the letter portion into `clarity_grade` (e.g. `'SI2'` → `'SI'`, `'IF'` → `'IF'`).
- Add `is_good_cut`: True where `cut` contains `'Good'`.
- Among `is_good_cut` diamonds, what is the mean `price` per `clarity_grade`? Sort descending.
- Convert `clarity_grade` to `category`. How many categories are there?

In [15]:
diamonds = sns.load_dataset('diamonds')

# Your code here

diamonds['clarity_grade'] = diamonds['clarity'].str.extract(r'([A-Z]+)')
diamonds['is_good_cut'] = diamonds['cut'].str.contains('Good')
diamonds.loc[diamonds['is_good_cut']].groupby('clarity_grade')['price'].mean().sort_values(ascending=False)

diamonds['clarity_grade'] = diamonds['clarity_grade'].astype('category')
len(diamonds['clarity_grade'].cat.categories)

5

---
## Level 2 — file paths

Parse `files` into structured columns, then answer the questions below. No hints on which patterns to use — figure out the regex yourself.

- Extract: `year`, `month`, `filename` (without extension), `ext` (e.g. `'csv'`).
- Which year has more files?
- Flag `is_versioned`: True where the filename contains a version pattern (e.g. `_v1`, `_v2`).
- Among versioned files, what is the highest version number? (Extract the version number and find the max.)

In [69]:
files = pd.DataFrame({'path': [
    '/data/2024/jan/sales_report_v2.csv',
    '/data/2024/feb/sales_report_v1.csv',
    '/data/2023/dec/inventory_final.xlsx',
    '/reports/2024/jan/summary_v3.pdf',
    '/data/2023/nov/sales_report_v1.csv',
    '/data/2024/mar/forecast_v1.csv',
    '/reports/2023/dec/summary_v2.pdf',
]})

# Your code here
#files['year'] = files['path'].str.extract(r'data/(\d+)')
splits = files['path'].str.split('/')
files['year'] = splits.str[2]
files['month'] = splits.str[3]
files['filename'] = splits.str[4].str.extract(r'(\w+)')
files['ext'] = splits.str[4].str.extract(r'\.(\w+)')

files['year'].value_counts().idxmax()
files['is_versioned'] = files['filename'].str.contains(r'_v\d+')

filesv =files[files['is_versioned']]

filesvn = filesv['filename'].str.extract(r'_v(\d+)')
filesvn.astype(int).max()


0    3
dtype: int64

---
## Level 3 — sales pipeline + MultiIndex

Each `record` encodes quarter, region, and revenue as a single string (e.g. `'Q1-NORTH-2450'`).

Write a `.pipe()` chain:
- **`parse(df)`** — extract `quarter`, `region`, and `revenue` (as int) from `record`.
- **`enrich(df)`** — add `revenue_tier`: `'low'` (< 2500), `'mid'` (2500–3499), `'high'` (≥ 3500). Add `rev_per_unit` = `revenue / units`.

After the chain:
- `groupby(['region', 'quarter'])['revenue'].sum()` — you'll get a MultiIndex Series. Call `.unstack()` on it. What shape is the result?
- Use `.xs()` to pull out just the `'Q1'` slice.
- Which region had the highest total revenue across all quarters?
- Build a dict comprehension `{region: mean_rev_per_unit}` from a groupby result.

In [68]:
sales = pd.DataFrame({
    'record':   ['Q1-NORTH-2450', 'Q2-SOUTH-1890', 'Q1-SOUTH-3100', 'Q3-NORTH-4200',
                 'Q2-NORTH-2750', 'Q3-SOUTH-1650', 'Q1-EAST-2900',  'Q2-EAST-3400',
                 'Q3-EAST-2100',  'Q1-WEST-3800',  'Q2-WEST-2200',  'Q3-WEST-4500'],
    'category': ['A', 'B', 'A', 'C', 'B', 'A', 'C', 'B', 'A', 'C', 'B', 'A'],
    'units':    [45, 30, 52, 38, 41, 28, 47, 55, 33, 62, 37, 58],
})

# Your code here
def parse(df):
    df['quarter'] = df['record'].str.extract(r'^(\w+)')
    df['region'] = df['record'].str.extract(r'-([A-Z]+)-')
    df['revenue'] = df['record'].str.extract(r'(\d+)$').astype(int)
    return df 

def enrich(df):
    df['revenue_tier'] = pd.cut(df['revenue'], bins= [0,2500,3500,np.inf],
                                labels=['low','mid','high'], right=False)
    df['rev_per_unit'] = df['revenue']/df['units']
    return df

res = sales.copy().pipe(parse).pipe(enrich)


ru = res.groupby(['region','quarter'])['revenue'].sum().unstack()
ru.shape

r = res.groupby(['region','quarter'])['revenue'].sum()
r.xs('Q1',level='quarter')

ru.sum(axis = 1).idxmax()

mr = res.groupby('region')['rev_per_unit'].mean()
{r:m for r,m in zip(mr.index, mr)}


{'EAST': 62.385557704706635,
 'NORTH': 77.34797698854182,
 'SOUTH': 60.51465201465201,
 'WEST': 66.11199631221878}

- `groupby(['region', 'quarter'])['revenue'].sum()` — you'll get a MultiIndex Series. Call `.unstack()` on it. What shape is the result?
- Use `.xs()` to pull out just the `'Q1'` slice.
- Which region had the highest total revenue across all quarters?
- Build a dict comprehension `{region: mean_rev_per_unit}` from a groupby result.

In [48]:
res

,record,category,units,quarter,region,revenue,revenue_tier,rev_per_unit
0,Q1-NORTH-2450,A,45,Q1,NORTH,2450,low,54.444444
1,Q2-SOUTH-1890,B,30,Q2,SOUTH,1890,low,63.000000
2,Q1-SOUTH-3100,A,52,Q1,SOUTH,3100,mid,59.615385
3,Q3-NORTH-4200,C,38,Q3,NORTH,4200,high,110.526316
4,Q2-NORTH-2750,B,41,Q2,NORTH,2750,mid,67.073171
5,Q3-SOUTH-1650,A,28,Q3,SOUTH,1650,low,58.928571
6,Q1-EAST-2900,C,47,Q1,EAST,2900,mid,61.702128
7,Q2-EAST-3400,B,55,Q2,EAST,3400,mid,61.818182
8,Q3-EAST-2100,A,33,Q3,EAST,2100,low,63.636364
9,Q1-WEST-3800,C,62,Q1,WEST,3800,high,61.290323
